In [3]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pandas as pd

df_03 = pd.read_csv("003_manchester_processed.csv")
df_04 = pd.read_csv("004_manchester_processed.csv")
df_06 = pd.read_csv("006_manchester_processed.csv")
df_08 = pd.read_csv("008_manchester_processed.csv")
# four predictd ataset
datasets = {
    "03": df_03,
    "04": df_04,
    "06": df_06,
    "08": df_08
}


In [4]:
# validation result list
val_results = []

# main
for name, df in datasets.items():
    print(f"\n========== Processing dataset {name} ==========")

    # split dataset
    train_df = df[(df['year'] >= 2006) & (df['year'] <= 2040)]
    valid_df = df[(df['year'] > 2040) & (df['year'] < 2050)]

    # predict dataset
    test_raw = df[(df['year'] >= 2050) & (df['year'] <= 2080)].copy()
    test_df = test_raw.copy()

    # delete unnecessary
    drop_cols = ['lat', 'lon', 'time', 'date']
    train_df = train_df.drop(columns=drop_cols, errors='ignore')
    valid_df = valid_df.drop(columns=drop_cols, errors='ignore')
    test_df  = test_df.drop(columns=drop_cols, errors='ignore')

    # target feature and label
    target = 'TREFMXAV_U'
    features = [col for col in train_df.columns if col != target]

    X_train, y_train = train_df[features], train_df[target]
    X_valid, y_valid = valid_df[features], valid_df[target]
    X_test,  y_test  = test_df[features],  test_df[target]

    # model
    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    # training
    xgb.fit(X_train, y_train)

    #  Validation
    y_valid_pred = xgb.predict(X_valid)

    val_mae = mean_absolute_error(y_valid, y_valid_pred)
    val_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_pred))

    print("Validation MAE:", val_mae)
    print("Validation RMSE:", val_rmse)

    # result
    val_results.append({
        "Dataset": name,
        "MAE": val_mae,
        "RMSE": val_rmse
    })

    # final training
    X_final = pd.concat([X_train, X_valid])
    y_final = pd.concat([y_train, y_valid])

    xgb.fit(X_final, y_final)

    # predict
    y_pred = xgb.predict(X_test)


    test_raw['prediction'] = y_pred

    # prediction csv
    filename = f"dataset_{name}_predictions.csv"
    test_raw.to_csv(filename, index=False)

    print(f"Saved: {filename}")




========== Processing dataset 03 ==========
Validation MAE: 0.5041381428162104
Validation RMSE: 0.6879836853731327
Saved: dataset_03_predictions.csv

========== Processing dataset 04 ==========
Validation MAE: 0.5159150504766241
Validation RMSE: 0.688045043504886
Saved: dataset_04_predictions.csv

========== Processing dataset 06 ==========
Validation MAE: 0.5217177720931919
Validation RMSE: 0.7107191532753695
Saved: dataset_06_predictions.csv

========== Processing dataset 08 ==========
Validation MAE: 0.5072534644062747
Validation RMSE: 0.690574886705961
Saved: dataset_08_predictions.csv


In [5]:
# output evaluation result
val_df = pd.DataFrame(val_results)

print("\n===== Validation Summary =====")
print(val_df)


===== Validation Summary =====
  Dataset       MAE      RMSE
0      03  0.504138  0.687984
1      04  0.515915  0.688045
2      06  0.521718  0.710719
3      08  0.507253  0.690575
